# Potato Contact — public baseline

This notebook contains a complete but simple baseline. It loads the public embeddings and proposes unused words that are close to the current winning word.

### Where should you make changes?

Your main work area is **Section 2: `PublicEmbeddingPlayer` — your solution**. That class decides which word to propose on every turn. You may rewrite its strategy, add methods, or change its parameters.

Sections 1 and 3 provide data loading and the contest input/output protocol. Run them, but normally do not change them. Section 4 explains how to export your solution.

## 1. Setup — run this cell, normally do not change it

This section loads `vocabulary.json` and `public_embeddings.npy` and prepares cosine similarities for the baseline.

In [2]:
import json
import os
import sys
from pathlib import Path

import numpy as np

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_DIR = Path(os.environ.get("POTATO_DATA_DIR", BASE_DIR / "dataset"))
DATA_DIR = Path('dataset/public')
WORDS_PATH = DATA_DIR / "vocabulary.json"
EMBEDDINGS_PATH = DATA_DIR / "public_embeddings.npy"

with WORDS_PATH.open() as file:
    words = json.load(file)

embeddings = np.load(EMBEDDINGS_PATH).astype(np.float32, copy=False)
if embeddings.ndim != 2 or embeddings.shape[0] != len(words):
    raise ValueError("Public embeddings are not aligned with the vocabulary")

norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1.0
normalized = embeddings / norms
similarities = normalized @ normalized.T
word_to_index = {word.casefold(): index for index, word in enumerate(words)}

print(
    f"Loaded {len(words)} words and embeddings with shape {embeddings.shape}",
    file=sys.stderr,
)
# the oracle uses cosine similarity, so while my earlier attempt at using euclidean distance might have helped
# the score a little bit when the state-tracking (tracking how many correct answers each word matches) 
# has not been implemented, maybe some of what's missing from my in-contest solution is to revert back to the
# cosine similarity approach when checking answers and finding best matching word :)
# also for the first few rounds if multiple words have the same perfect match, probably fall back to that 
# embedding distance binary-search method (try to halve the space) until you narrow down the 
# search space to around 8 to 16 words, then you guess and gather questions one by one

Loaded 1602 words and embeddings with shape (1602, 2560)


## 2. `PublicEmbeddingPlayer` — your solution

**This is the main cell you are expected to improve.** Everything in this cell is part of the baseline strategy.

You may replace the whole strategy, but keep these two parts of the interface:

1. the class must still be named `PublicEmbeddingPlayer`;
2. `respond(message)` must return one word from the provided vocabulary.

---

This code block below implements a historical constraint-satisfaction heuristic that tracks past round outcomes to filter and rank candidate words:

* **State Tracking:** Records winning and losing word indices for every turn to build candidate constraints.
* **Agreement Scoring:** Calculates directional similarity differences ($\text{sim}_{\text{winner}} - \text{sim}_{\text{loser}}$) and computes a binary agreement sum (`lst`) to measure how consistently candidates align with all past match results.
* **Priority Ranking:** Boosts candidates that satisfy 100% of historical conditions to top priority (`100.0`), while maintaining secondary scores (based on the total sum of directional similarity differences) for non-perfect matches to ensure robust fallback selection.
* **Deduplication:** Iterates through the sorted candidates in descending order of similarity, returning the top unproposed word.

In [6]:
#### ===================== YOUR SOLUTION STARTS HERE =====================
# Change the parameters, methods, or the whole strategy in this cell.
import json
import sys
import numpy as np
class PublicEmbeddingPlayer:
    def __init__(self):
        self.proposed = set()
        # track state
        self.winners = []
        self.losers = []

    def respond(self, message):
        champion = word_to_index[message["winner_word"].casefold()]
        w1 = word_to_index[message["word1"].casefold()]
        w2 = word_to_index[message["word2"].casefold()]
        if w1 == champion:
            loser = w2
        else:
            loser = w1
        self.winners.append(champion)
        self.losers.append(loser)
        lst = [np.sign(similarities[c] - similarities[l]) for c, l in zip(self.winners, self.losers)]
        sim = [similarities[c] - similarities[l] for c, l in zip(self.winners, self.losers)]
        lst = np.array(lst).sum(axis=0)
        sim = np.array(sim).sum(axis=0)
        # similarities[champion] - similarities[loser] alone resulted in 51.45 local score, 21.47 lb B score 
        # 74/120 local win rate, up from 51/120 local win rate
        # Pick the next unused word most similar to the winner
        # have it track state (np.sign to track how many it got right): 95.07 local score, 39.88 lb B score
        matches = (lst == len(self.winners)).sum()
        sim[lst == len(self.winners)] = 100.0
        # sim[lst != len(self.winners)] = 0.0
        # what has happened here (this scored so high, 94.97 local, 53.97 lb A, 47.02 lb B) 
        # ok if i uncomment the line of code 2 lines ago the lb A score drops to 10.87
        # so the mechanism for handling imperfect matches (all perfect matches alr used) is surprisingly important !!
        # unfortunately this task is super heavy on experimentation
        # local validation also doesn't match super well with lb A/B score with my attempts so there might be some luck
        # involved given only 15 submissions
        for index in np.argsort(-sim):
            index = int(index)
            if index not in self.proposed:
                self.proposed.add(index)
                return words[index]
        return words[0]

# ====================== YOUR SOLUTION ENDS HERE ======================

## 3. Contest protocol — run this cell, do not change it

This section connects your `PublicEmbeddingPlayer` to the judging system. Your program runs **once** and plays every game: it reads one JSON message per line, makes a fresh `PublicEmbeddingPlayer` at each `{"event": "new_game"}`, calls `player.respond(message)` on each turn, and exits on `{"event": "done"}`.

You normally should not edit this cell. Diagnostic output goes to `stderr`, so it does not interfere with the protocol. The final condition starts the loop only after the notebook has been exported to a `.py` file; running this cell inside Jupyter will not wait for terminal input.

In [4]:
def run_interactive():
    # Your program runs ONCE and plays every game. Preparation (Section 1) already
    # ran above. A fresh PublicEmbeddingPlayer is created for each new game so its
    # per-game state resets, while the loaded embeddings/similarities are reused.
    player = PublicEmbeddingPlayer()
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue

        try:
            message = json.loads(line)
        except json.JSONDecodeError:
            continue

        event = message.get("event")
        if event == "done":            # all games finished -> exit
            break
        if event == "new_game":        # a new game starts -> reset per-game state
            player = PublicEmbeddingPlayer()
            continue
        if "status" in message:        # this game ended (win/loss) -> wait for the next
            continue

        new_word = player.respond(message)
        print(
            f"turn={message['turn']} winner={message['winner_word']} proposal={new_word}",
            file=sys.stderr,
        )
        print(json.dumps({"new_word": new_word}), flush=True)


if "__file__" in globals():
    run_interactive()


## 4. Test and submit your solution

When you are happy with your changes in `PublicEmbeddingPlayer`:

1. Save `solution.ipynb`; this saved notebook is your official Contest submission;
2. Add the saved `solution.ipynb` to git, commit and push.  
3. Submit it through the Contest interface; it should be visible as the last commit.
4. Done — it will be tested on the private test set; you can see the result once it finishes.




Optionally, you can check the solution locally
```
python local_test.py solution.ipynb
```
The local tester uses public embeddings, so its score is approximate and may differ from the official private score. Do not add ordinary `print(...)` calls to standard output: the judge expects protocol JSON there. Send debugging messages to standard error with `print(..., file=sys.stderr)`.

In [7]:
# in an external shell, run this command:
# python local_test.py solution.ipynb (this one is local validation)
# python local_test_lb_a.py solution.ipynb (this one is for LB A score) (baseline: 15.00)
# python local_test_lb_b.py solution.ipynb (this one is for LB B score) (baseline: 17.08)
# note that you cannot uncomment the above line as it would then try to recursively call itself, causing an error
